# Wallet Strategy Selection

Unified selection stage: runs **all wallet-selection methods** and saves each
as a named `WalletSelectionStrategy` artifact in the workspace.  The backtest
stage loads these artifacts directly — no strategy logic is re-built there.

## Methods included
| id suffix | description |
|-----------|-------------|
| `skill_sweep` → `quality_core` | top-N by best skill metric (grid-searched on train-a→train-b) |
| `cohort_early_entry` | wallets that enter markets early |
| `cohort_smooth_pnl` | wallets with high PnL relative to volatility |
| `volatility` | volatility-filtered wallets (from the profitable_wallet_analysis path) |

Each method produces **two trigger variants**:
- `score_threshold` — composite signal score ≥ calibrated threshold (Kelly sizing)
- `all_open_buys`   — every open-buy event (fixed stake baseline)


In [1]:
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds


## Configuration

In [2]:
# ── configuration ─────────────────────────────────────────────────────────────
RECENCY_DAYS     = 90

PRICE_BUCKET_BINS   = [0.0, 0.1, 0.25, 0.4, 0.6, 0.75, 0.9, 1.0]
PRICE_BUCKET_LABELS = [
    "0.00-0.10", "0.10-0.25", "0.25-0.40", "0.40-0.60",
    "0.60-0.75", "0.75-0.90", "0.90-1.00",
]

TRADES_DIR    = Path("../../data/polygon_trades_processed")
WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

dataset = ds.dataset(TRADES_DIR, format="parquet")
DATASET_COLUMNS        = set(dataset.schema.names)
TRIGGER_TX_HASH_COLUMN = (
    "transaction_hash" if "transaction_hash" in DATASET_COLUMNS
    else ("tx_hash" if "tx_hash" in DATASET_COLUMNS else None)
)

METRICS_A_PATH          = WORKSPACE_DIR / "wallet_metrics_train_a.parquet"
METRICS_B_PATH          = WORKSPACE_DIR / "wallet_metrics_train_b.parquet"
METRICS_FULL_PATH       = WORKSPACE_DIR / "wallet_metrics_full_train.parquet"
METRICS_TEST_PATH       = WORKSPACE_DIR / "wallet_metrics_test.parquet"
CALIBRATION_SIGNALS_PATH = WORKSPACE_DIR / "signal_events_train_b.parquet"
TEST_SIGNALS_PATH       = WORKSPACE_DIR / "signal_events_test.parquet"


## Derive train/test dates from data

In [3]:
# ── derive train/test boundary from is_train flag ───────────────────────────
_sample = pd.read_parquet(sorted(TRADES_DIR.glob("*.parquet"))[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN   = _sample.loc[_sample["is_train"], "dt"].max().date()
TRAIN_START_DATE = _sample.loc[_sample["is_train"], "dt"].min().date()
TRAIN_A_END_DATE = TRAIN_START_DATE + (END_DATE_TRAIN - TRAIN_START_DATE) // 2
del _sample
print(f"END_DATE_TRAIN={END_DATE_TRAIN}  TRAIN_A_END_DATE={TRAIN_A_END_DATE}")


END_DATE_TRAIN=2026-02-15  TRAIN_A_END_DATE=2025-08-18


## Compute / load wallet skill metrics

In [4]:
from polymarket_analysis.wallet_selection.metrics import compute_wallet_skill_workspace

if all(p.exists() for p in [METRICS_A_PATH, METRICS_B_PATH, METRICS_FULL_PATH, METRICS_TEST_PATH]):
    train_a_metrics    = pd.read_parquet(METRICS_A_PATH)
    train_b_metrics    = pd.read_parquet(METRICS_B_PATH)
    full_train_metrics = pd.read_parquet(METRICS_FULL_PATH)
    test_metrics       = pd.read_parquet(METRICS_TEST_PATH)
else:
    train_a_metrics, train_b_metrics, full_train_metrics, test_metrics = (
        compute_wallet_skill_workspace(
            dataset,
            end_date_train=END_DATE_TRAIN,
            train_a_end_date=TRAIN_A_END_DATE,
            recency_days=RECENCY_DAYS,
        )
    )
    train_a_metrics.to_parquet(METRICS_A_PATH, index=False)
    train_b_metrics.to_parquet(METRICS_B_PATH, index=False)
    full_train_metrics.to_parquet(METRICS_FULL_PATH, index=False)
    test_metrics.to_parquet(METRICS_TEST_PATH, index=False)

pd.Series({
    "train_a_wallets":    len(train_a_metrics),
    "train_b_wallets":    len(train_b_metrics),
    "full_train_wallets": len(full_train_metrics),
    "test_wallets":       len(test_metrics),
}).to_frame("value")


,value
train_a_wallets,45407
train_b_wallets,45407
full_train_wallets,45407
test_wallets,45407


## Cohort selection sweep (skill path)

Grid-search over metrics × top-N using train-a → train-b persistence.

In [5]:
from polymarket_analysis.wallet_selection.selector import (
    CANDIDATE_METRICS,
    cohort_selection_sweep,
    select_wallets,
)

cohort_sweep = cohort_selection_sweep(train_a_metrics, train_b_metrics, CANDIDATE_METRICS)
cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge"], ascending=False
).head(20)


,metric,top_n,wallets_found_in_train_b,train_b_open_buy_trades,train_b_weighted_prob_edge,train_b_avg_prob_edge,train_b_avg_copy_roi_capped,train_b_total_copy_pnl_usdc,train_b_hit_rate
0,prob_edge_shrunk,50,50,2188,0.084714,0.144695,0.901704,100566.740631,0.417733
1,prob_edge_shrunk,100,100,3418,-0.005363,0.103747,0.597762,32786.499393,0.447630
2,prob_edge_shrunk,200,200,10178,0.000995,0.084642,0.438224,72766.808891,0.470132
13,avg_copy_roi_capped,300,300,9398,0.030304,0.066977,0.410752,174619.921275,0.433922
12,avg_copy_roi_capped,200,200,4473,0.036930,0.059738,0.384170,128404.802915,0.418735
3,prob_edge_shrunk,300,300,13101,0.012600,0.079378,0.370006,166870.171841,0.496756
14,avg_copy_roi_capped,500,500,13994,0.027761,0.042974,0.324798,260347.157546,0.454838
10,avg_copy_roi_capped,50,50,2141,0.015695,0.051842,0.307529,19774.525120,0.410556
11,avg_copy_roi_capped,100,100,2705,0.025123,0.047399,0.266803,51251.827690,0.419593
4,prob_edge_shrunk,500,500,26621,0.029551,0.061264,0.222028,640161.293125,0.513805


In [6]:
# pick best metric / top-N
best_row = cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge", "train_b_open_buy_trades"],
    ascending=[False, False, False],
).iloc[0]
BEST_SELECTION_METRIC = best_row["metric"]
BEST_TOP_N            = int(best_row["top_n"])
{"best_metric": BEST_SELECTION_METRIC, "best_top_n": BEST_TOP_N}


{'best_metric': 'prob_edge_shrunk', 'best_top_n': 50}

## Select wallets (skill path) + build cohorts

In [7]:
selected_wallets = select_wallets(full_train_metrics, BEST_SELECTION_METRIC, BEST_TOP_N)
selected_wallets.to_parquet(WORKSPACE_DIR / "selected_wallets_v2.parquet", index=False)
selected_wallets[[
    "wallet", "open_buy_trades", "distinct_markets",
    "recent_open_buy_trades", BEST_SELECTION_METRIC, "wallet_quality"
]].head(15)


,wallet,open_buy_trades,distinct_markets,recent_open_buy_trades,prob_edge_shrunk,wallet_quality
0,0xbb0bd109b9f0c2a59b8819c466f064cf65ab3790,532,506,529,0.304533,1.00
1,0x5ad5c4608c4661361b91c92e1091d2c5b43c37b9,772,743,769,0.300616,0.98
2,0x27b713fc1c89d498f19977c8bc7788a161fb7710,787,23,787,0.294186,0.96
3,0x4d7fad0c5944fc24d4a67110f8e31abd5f559485,218,210,170,0.290809,0.94
4,0x54a2c4cfc4332d831acc3f5a860d6540982c1d43,208,199,208,0.285536,0.92
5,0x82307f44c9405e73dc1cff466073dcc505535121,389,368,389,0.261187,0.90
6,0xb9bca9fa4069228a37b583d64ff75efdf36b3498,21,21,21,0.248022,0.88
7,0xf1f06f49be8ce5681752ae80e660aeaace6858df,308,302,92,0.235090,0.86
8,0xc178402031235263f78c1a43bba8cd49d2be35b3,119,114,12,0.220636,0.84
9,0xfd4263b3ad08226034fe1b1ea678a46d80b58895,209,203,209,0.219044,0.82


## Build wallet profiles and signal events

In [8]:
from polymarket_analysis.signal.builder import (
    build_wallet_profiles,
    build_signal_events,
)

selected_wallet_profiles = build_wallet_profiles(
    dataset, selected_wallets, period="full_train",
    end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE
)
selected_wallet_profiles.to_parquet(
    WORKSPACE_DIR / "selected_wallet_profiles_v2.parquet", index=False
)

if CALIBRATION_SIGNALS_PATH.exists():
    train_b_open_buys = pd.read_parquet(CALIBRATION_SIGNALS_PATH)
else:
    _, train_b_open_buys = build_signal_events(
        dataset, selected_wallet_profiles, period="train_b",
        end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
        price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
        tx_hash_column=TRIGGER_TX_HASH_COLUMN,
    )
    train_b_open_buys.to_parquet(CALIBRATION_SIGNALS_PATH, index=False)

if TEST_SIGNALS_PATH.exists():
    test_open_buys = pd.read_parquet(TEST_SIGNALS_PATH)
else:
    _, test_open_buys = build_signal_events(
        dataset, selected_wallet_profiles, period="test",
        end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
        price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
        tx_hash_column=TRIGGER_TX_HASH_COLUMN,
    )
    test_open_buys.to_parquet(TEST_SIGNALS_PATH, index=False)

print(f"train_b signals: {len(train_b_open_buys):,}  test signals: {len(test_open_buys):,}")


train_b signals: 2,188  test signals: 2,936


## Calibrate signal scoring on train-B

In [9]:
from polymarket_analysis.signal.scorer import (
    build_calibration_tables,
    apply_signal_score,
    select_signal_threshold,
)

price_table, consensus_table, global_edge = build_calibration_tables(train_b_open_buys)
train_b_scored = apply_signal_score(train_b_open_buys, price_table, consensus_table)
SIGNAL_THRESHOLD = select_signal_threshold(train_b_scored)
print(f"Global edge: {global_edge:.4f}")
print(f"Selected signal threshold: {SIGNAL_THRESHOLD:.2f}")

# score distribution
score_deciles = (
    train_b_scored.assign(
        score_decile=lambda df: pd.qcut(df["signal_score"], q=10, labels=False, duplicates="drop")
    )
    .groupby("score_decile")
    .agg(
        count=("signal_score", "size"),
        avg_signal_score=("signal_score", "mean"),
        avg_copy_roi_capped=("copy_roi_capped", "mean"),
    )
    .reset_index()
)
score_deciles


Global edge: 0.1447
Selected signal threshold: 0.80


,score_decile,count,avg_signal_score,avg_copy_roi_capped
0,0,219,0.444743,1.860643
1,1,219,0.501713,1.754792
2,2,219,0.576948,1.051924
3,3,218,0.626133,0.362047
4,4,219,0.659588,0.493976
5,5,219,0.697506,0.589540
6,6,226,0.740450,0.394098
7,7,214,0.784169,0.830189
8,8,216,0.840562,1.673228
9,9,219,0.931566,0.029301


## Build wallet cohorts (skill path)

In [10]:
from polymarket_analysis.wallet_selection.selector import build_wallet_cohorts

wallet_cohorts = build_wallet_cohorts(
    full_train_metrics, train_b_open_buys, selected_wallets,
    top_n=BEST_TOP_N,
)
{name: len(df) for name, df in wallet_cohorts.items()}


{'quality_core': 50, 'early_entry': 32, 'smooth_pnl': 50}

## Volatility-based wallet selection (second method)

Loads the full training set, applies the volatility filter, and stores the
result as an additional `WalletSelectionStrategy`.  The volatility wallet set
is added to `wallet_cohorts` so the strategy factory handles it uniformly.

In [73]:
from polymarket_analysis.wallet_selection.volatility import (
    compute_wallet_metrics,
    filter_wallets_by_volatility,
)

# Load full training trades for the volatility path
trade_files = sorted(TRADES_DIR.glob("*.parquet"))
df_train_vol = pd.concat([pd.read_parquet(f) for f in trade_files], ignore_index=True)
df_train_vol["dt"] = pd.to_datetime(df_train_vol["dt"], utc=True)
df_train_vol["usdc_amount"]      = df_train_vol["usdc_amount"].astype(float)
df_train_vol["final_value_usdc"] = df_train_vol["final_value_usdc"].astype(float)
df_train_vol["quantity"]         = df_train_vol["quantity"].astype(float)

# PnL and notional per fill
df_train_vol["pnl"] = np.where(
    df_train_vol["side"] == "BUY",
    df_train_vol["final_value_usdc"] - df_train_vol["usdc_amount"],
    df_train_vol["usdc_amount"] - df_train_vol["final_value_usdc"],
)
df_train_vol["notional"] = np.where(
    df_train_vol["side"] == "BUY",
    df_train_vol["usdc_amount"],
    df_train_vol["quantity"] * (1 - df_train_vol["price"].astype(float)),
)
df_slice = df_train_vol[df_train_vol["is_train"]].copy()

wallet_vol_train, _ = compute_wallet_metrics(df_slice)
del df_train_vol, df_slice

In [80]:
filtered_wallets_vol = filter_wallets_by_volatility(
    wallet_vol_train,
    min_buckets=20,
    max_top5_pnl_pct=0.4,
    max_top_market_pnl_pct=0.5,
)

filtered_wallets_vol = filtered_wallets_vol[filtered_wallets_vol['average_roi'] >= 0.04].sort_values("pnl_volatility", ascending=True).head(100)

print(f"Volatility-filtered wallets: {len(filtered_wallets_vol)}")

# Build wallet_quality based on total_pnl rank (higher = better)
filtered_wallets_vol = filtered_wallets_vol.copy().reset_index(drop=True)
filtered_wallets_vol["wallet_quality"] = filtered_wallets_vol["total_pnl"].rank(
    method="first", pct=True
)

# Add as additional cohort (uses only train-b signals for trigger calibration)
# We intersect with wallets that have signals to ensure coverage
vol_wallets_with_signals = set(train_b_open_buys["wallet"]).intersection(
    set(filtered_wallets_vol["wallet"])
)
wallet_cohorts["volatility"] = filtered_wallets_vol.copy().reset_index(drop=True)

# [
#     filtered_wallets_vol["wallet"].isin(vol_wallets_with_signals)
# ][["wallet", "wallet_quality"]].copy().reset_index(drop=True)

print(f"Volatility cohort: {len(wallet_cohorts['volatility'])}")

Volatility-filtered wallets: 100
Volatility cohort: 100


## Build and save strategy registry

All cohorts × all trigger variants → persisted `WalletSelectionStrategy` files.
The backtest loads these directly.

In [82]:
from polymarket_analysis.wallet_selection.selector import build_strategies_from_sweep
from polymarket_analysis.strategy.registry import save_strategy

all_strategies = build_strategies_from_sweep(
    wallet_cohorts=wallet_cohorts,
    signal_threshold=SIGNAL_THRESHOLD,
    selection_metric=BEST_SELECTION_METRIC,
    top_n=BEST_TOP_N,
    sweep_df=cohort_sweep,
    extra_metadata={
        "end_date_train": str(END_DATE_TRAIN),
        "train_a_end_date": str(TRAIN_A_END_DATE),
    },
)

for strategy in all_strategies:
    parquet_path, json_path = save_strategy(strategy, WORKSPACE_DIR)
    print(f"Saved [{strategy.strategy_id}]  wallets={len(strategy.wallets):3d}  trigger={strategy.trigger.fn_ref.split('.')[-1]}")

print(f"\nTotal strategies saved: {len(all_strategies)}")


Saved [quality_core__score_threshold]  wallets= 50  trigger=score_threshold
Saved [quality_core__all_open_buys]  wallets= 50  trigger=all_open_buys
Saved [early_entry__score_threshold]  wallets= 32  trigger=score_threshold
Saved [early_entry__all_open_buys]  wallets= 32  trigger=all_open_buys
Saved [smooth_pnl__score_threshold]  wallets= 50  trigger=score_threshold
Saved [smooth_pnl__all_open_buys]  wallets= 50  trigger=all_open_buys
Saved [volatility__score_threshold]  wallets=100  trigger=score_threshold
Saved [volatility__all_open_buys]  wallets=100  trigger=all_open_buys

Total strategies saved: 8


## Strategy registry summary

In [83]:
from polymarket_analysis.strategy.registry import load_all_strategies

registry = load_all_strategies(WORKSPACE_DIR)
summary_rows = []
for sid, s in registry.items():
    summary_rows.append({
        "strategy_id": s.strategy_id,
        "name": s.name,
        "selection_method": s.selection_method,
        "num_wallets": len(s.wallets),
        "trigger_fn": s.trigger.fn_ref.split(".")[-1],
        "threshold": s.trigger.params.get("threshold"),
        "dynamic_sizing": s.trigger.params.get("dynamic_sizing"),
    })

pd.DataFrame(summary_rows)


,strategy_id,name,selection_method,num_wallets,trigger_fn,threshold,dynamic_sizing
0,early_entry__all_open_buys,early_entry | all open-buys (fixed stake),cohort_early_entry,32,all_open_buys,NaN,False
1,early_entry__score_threshold,early_entry | score >= 0.80 (Kelly),cohort_early_entry,32,score_threshold,0.8,True
2,quality_core__all_open_buys,quality_core | all open-buys (fixed stake),skill_sweep,50,all_open_buys,NaN,False
3,quality_core__score_threshold,quality_core | score >= 0.80 (Kelly),skill_sweep,50,score_threshold,0.8,True
4,smooth_pnl__all_open_buys,smooth_pnl | all open-buys (fixed stake),cohort_smooth_pnl,50,all_open_buys,NaN,False
5,smooth_pnl__score_threshold,smooth_pnl | score >= 0.80 (Kelly),cohort_smooth_pnl,50,score_threshold,0.8,True
6,volatility__all_open_buys,volatility | all open-buys (fixed stake),volatility,100,all_open_buys,NaN,False
7,volatility__score_threshold,volatility | score >= 0.80 (Kelly),volatility,100,score_threshold,0.8,True


## Wallet PnL over time

Two-panel plot for every selection cohort (including volatility):
- **Panel 1** — individual cumulative PnL lines for the top wallets of each cohort
- **Panel 2** — aggregate cumulative PnL of each cohort

Set `PLOT_WALLET_PNL = False` to skip (e.g. when running headlessly).

In [84]:
PLOT_WALLET_PNL  = True
TOP_N_INDIVIDUAL = 20   # wallets shown in panel 1 per cohort


In [85]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [87]:
_trade_files = sorted(TRADES_DIR.glob("*.parquet"))
df_fills = pd.concat([pd.read_parquet(f) for f in _trade_files], ignore_index=True)
df_fills["dt"]               = pd.to_datetime(df_fills["dt"], utc=True)
df_fills["usdc_amount"]      = df_fills["usdc_amount"].astype(float)
df_fills["final_value_usdc"] = df_fills["final_value_usdc"].astype(float)
df_fills["quantity"]         = df_fills["quantity"].astype(float)

In [95]:
df_fills.head()

,wallet,condition_id,dt,side,token_id,outcome,quantity,price,usdc_amount,position,final_value_usdc,trade_pnl,token_winner,final_price,tx_hash,is_train
0,0x011f2d377e56119fb09196dffb0948ae55711122,0x0c39af434de02f0e9d29f9d40b7ddfd70b000c37cde8...,2025-02-19 21:22:31+00:00,BUY,8673812547162584472287110199414147728954339962...,Girl,8.55,0.5,4.275,8.55,0.0,-4.275,False,0.0,0xee281249f2ef2f1f2a5c2e3313bcd4bd89cc18192ebe...,True
1,0x25257a6a89dba93dd0c536b6279365632a4eb919,0x0c39af434de02f0e9d29f9d40b7ddfd70b000c37cde8...,2025-02-19 21:22:31+00:00,BUY,8673812547162584472287110199414147728954339962...,Girl,41.45,0.5,20.725,41.45,0.0,-20.725,False,0.0,0xee281249f2ef2f1f2a5c2e3313bcd4bd89cc18192ebe...,True
2,0xa8c63f775ddbbe66b56614191747def3021444e8,0x0c39af434de02f0e9d29f9d40b7ddfd70b000c37cde8...,2025-02-20 03:46:08+00:00,BUY,8673812547162584472287110199414147728954339962...,Girl,20.00,0.5,10.000,20.00,0.0,-10.000,False,0.0,0x7779fd9fda7da5c2d6feb51eca0f02224dcf03df0302...,True
3,0xa8c63f775ddbbe66b56614191747def3021444e8,0x0c39af434de02f0e9d29f9d40b7ddfd70b000c37cde8...,2025-02-20 04:12:34+00:00,BUY,8673812547162584472287110199414147728954339962...,Girl,3.00,0.5,1.500,23.00,0.0,-1.500,False,0.0,0x6d41b21ca5b36f5eb275e75f52505409146442ec8693...,True
4,0xa8c63f775ddbbe66b56614191747def3021444e8,0x0c39af434de02f0e9d29f9d40b7ddfd70b000c37cde8...,2025-02-20 10:00:08+00:00,BUY,8673812547162584472287110199414147728954339962...,Girl,50.00,0.5,25.000,73.00,0.0,-25.000,False,0.0,0xdab4fc4b9008905fae1bc4e996594b78e09d58a16a11...,True


In [88]:
len(df_fills)

20808775

In [96]:
df_grouped = df_fills.groupby(['dt', 'wallet'])

In [93]:
wallet_cohorts['early_entry']['wallet']

0     0xc178402031235263f78c1a43bba8cd49d2be35b3
1     0x71cd52a9bf9121cf8376ba13999468f5d659912d
2     0x083b9f46b0d8e3d1b5ed0863c68f37df20efa478
3     0xb21f4dcec9d8a0cd71328a0c379b1e71a91e60d6
4     0x7a6192ea6815d3177e978dd3f8c38be5f575af24
5     0x4bbfdd0d844bc5e1c97e4049162578d6e7a8a331
6     0xfb81f27f1c8758d477332f8e751322c424da1cf3
7     0x1aa4520f779ca033fb2b3f16eee53882bb532b34
8     0xf989bd9c62b1eae2c388515fcc766527a8b147cc
9     0x5bbefc673462f1955e31b4a2347450724946c65d
10    0xc9b6227a295985591fe576ff2e054267a78a9b6a
11    0xdecc6bfa0303eb8e218fcb7c0b67e12902052bab
12    0xfdbad1dd309af982893b0d0b22340e398a7c9829
13    0x66b64985f12db6e7aff93345e32fb2dd2e2b1c3e
14    0x038ecb910d2c16ea5257db15c1a5f48d32291d43
15    0xa8cce5524b0c9664300002f4d1f779c41c3b728f
16    0x7a69709f4fce9acecc06f21bc7b3efdb4a200195
17    0x68a753b7e3119ea8556adf52783ab6b8ef5f46df
18    0x2cde80346eed526242e266476cfcc65073635b98
19    0xfe15ab2b5d5af545aecf26e620dd32b9890f865b
20    0x03626d34381b

In [ ]:
from polymarket_analysis.visualization.wallet_plots import plot_wallet_selection_pnl

if PLOT_WALLET_PNL:
    _trade_files = sorted(TRADES_DIR.glob("*.parquet"))
    df_fills = pd.concat([pd.read_parquet(f) for f in _trade_files], ignore_index=True)
    df_fills["dt"]               = pd.to_datetime(df_fills["dt"], utc=True)
    df_fills["usdc_amount"]      = df_fills["usdc_amount"].astype(float)
    df_fills["final_value_usdc"] = df_fills["final_value_usdc"].astype(float)
    df_fills["quantity"]         = df_fills["quantity"].astype(float)

    split_dt = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

    fig = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        top_n_individual=TOP_N_INDIVIDUAL,
        title="Wallet selection cohorts — cumulative PnL (train + test)",
    )
    fig.show(renderer="browser")
    del df_fills
else:
    print("PLOT_WALLET_PNL=False — skipping plots")
